# Assessing EUMETSAT Metop-SG Products for Cloud-native Access


**Authors**: Rajat Shinde (UAH), Harshini Girish (UAH), Alex Mandel (Development Seed), Brian Freitag (NASA MSFC)

**Date**: September 16, 2026

**Description**: Metop-SGA1 carries six instrument missions: METimage (VII),
IASI-NG, MWS, a Radio Occultation sounder, 3MI, and the Copernicus
Sentinel-5/UVNS spectrometer. Their products are distributed through the
EUMETSAT Data Store as zip packages containing a data file and two XML
sidecars. This notebook checks, for one recent granule per instrument, how
close each product is to cloud-native access.

**Setup**: This notebook will:

1. Find the Data Store collection for each instrument
2. Check whether the hosting answers HTTP range requests
3. Open a granule lazily over HTTP with xarray, without downloading it
4. Download one granule and read its chunking and compression settings
5. Build a kerchunk index and read the remote file through the Zarr engine

## Run this notebook

You need a free EUMETSAT account. Register at
[user.eumetsat.int](https://user.eumetsat.int), then copy your consumer key
and secret from [api.eumetsat.int/api-key](https://api.eumetsat.int/api-key).
The token they produce expires after about an hour; re-run the credentials
cell if requests start returning 401.

In [1]:
%pip install -q eumdac "xarray>=2024.10" h5netcdf fsspec aiohttp kerchunk zarr pandas requests zstandard

Note: you may need to restart the kernel to use updated packages.


In [44]:
import json, shutil, urllib.parse, zlib
from pathlib import Path
import numpy as np, pandas as pd, requests, fsspec, xarray as xr, eumdac

KEY, SECRET = Path.home().joinpath(".eumdac", "credentials").read_text().strip().split(",")
token = eumdac.AccessToken((KEY, SECRET))
store = eumdac.DataStore(token)

def auth():
    return {"Authorization": f"Bearer {token}"}   # str(token) auto-refreshes

DATA = Path("data"); DATA.mkdir(exist_ok=True)
print("token ok")

token ok


## About the datasets

The six products assessed here are the ones listed on the
[Metop-SG test data page](https://user.eumetsat.int/resources/user-guides/metop-sg-test-data):
one per instrument on Metop-SGA1.

| Instrument | Measures | Product |
|---|---|---|
| METimage (VII) | visible and infrared radiances, 20 channels | level 1B radiances |
| MWS | microwave sounding, 24 channels | level 1B |
| Radio Occultation (GRAS-2) | GNSS bending angles | level 1B |

They come through two different doors, and the notebook handles both:

- **Data Store collections** exist for the instruments already distributing
  flight data. As of this writing that is METimage (`EO:EUM:DAT:0464`),
  MWS (`EO:EUM:DAT:0450`) and GRAS-2 radio occultation (`EO:EUM:DAT:0452`).
- **Test-data downloads** are direct links on the page above, for the
  instruments not yet in the Data Store (IASI-NG, 3MI, Sentinel-5) and as
  pre-launch samples for the others. Open the page in a browser, copy each
  product's download link, and paste it below.

The distinction is itself part of the assessment: a test-data link tells you
about the file format EUMETSAT intends to ship, while only a Data Store
collection tells you about the hosting the operational data will live behind.


In [28]:
# Each product is either a Data Store collection ID or a direct download
# link copied from the Metop-SG test data page. Fill in the missing links.
PRODUCTS = {
    "METimage (VII)":    {"collection": "EO:EUM:DAT:0464"},
    "MWS":               {"collection": "EO:EUM:DAT:0450"},
    "Radio Occultation": {"collection": "EO:EUM:DAT:0452"},
}

## Helper functions

Five short functions used in every section. `latest` fetches the newest
product in a collection and lists the files inside its delivery package.
`ranges_ok` sends two ranged GETs; the suffix range matters because zip
central directories and Parquet footers sit at the end of the file, so a
server that cannot serve file tails cannot serve those formats in place.
`open_remote` opens a URL with xarray through fsspec, reading only the bytes
h5netcdf asks for. `layout` tabulates chunk shape and codec per variable.
`fetch` downloads a direct test-data link. `try_kerchunk` indexes the chunks of a local copy, points the references at
the remote URL, and reads one slice back through the Zarr engine.

In [47]:
enc = lambda s: urllib.parse.quote(s, safe="")

def latest(collection_id):
    """Newest product in a collection, plus the files in its package."""
    prod = store.get_collection(collection_id).search().first()
    entries = list(prod.entries)
    print(prod, "\n  files:", entries)
    return prod, entries

def entry_url(prod, filename):
    """Direct URL for one file inside the product, bypassing the zip."""
    return f"{prod.url.split('?')[0]}/entry?name={enc(filename)}"

def ranges_ok(url):
    """True when the server answers 206 for both a leading and a suffix range."""
    codes = {}
    for label, rng in (("head", "bytes=0-1023"), ("tail", "bytes=-65536")):
        r = requests.get(url, headers={**auth(), "Range": rng}, stream=True, timeout=60)
        codes[label] = r.status_code
        if label == "head":
            print("first bytes:", r.raw.read(8).hex(), " (894844... means HDF5/netCDF-4)")
        r.close()
    print("range status:", codes)
    return codes["head"] == 206 and codes["tail"] == 206

def fetch(url, filename=None):
    """Download a direct test-data link. Zips need unzipping afterwards."""
    dest = DATA / (filename or url.split("/")[-1].split("?")[0])
    if not dest.exists():
        with requests.get(url, headers=auth(), stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(dest, "wb") as f:
                shutil.copyfileobj(r.raw, f)
    print(f"{dest.name}: {dest.stat().st_size/1e6:.0f} MB")
    return dest

def auth():
    return {"Authorization": f"Bearer {token}"}   # str(token) auto-refreshes
    
def remote_size(url):
    """File size via a 1-byte ranged GET, since the endpoint may not do HEAD."""
    r = requests.get(url, headers={**auth(), "Range": "bytes=0-0"},
                     stream=True, timeout=60)
    r.close()
    if r.status_code == 206:
        return int(r.headers["Content-Range"].split("/")[-1])
    return int(r.headers["Content-Length"])

def open_remote(url):
    """Lazy-open a remote netCDF-4 over HTTP. Downloads nothing."""
    fs = fsspec.filesystem("https", client_kwargs={"headers": auth()},
                           encoded=True, skip_instance_cache=True)
    return xr.open_datatree(fs.open(url, block_size=4 * 2**20, size=remote_size(url)),
                            engine="h5netcdf", phony_dims="access",
                            decode_times=False)

def download(prod, filename):
    """Fetch one file from the product through the entry endpoint."""
    dest = DATA / filename
    if not dest.exists():
        with prod.open(entry=filename) as src, open(dest, "wb") as dst:
            shutil.copyfileobj(src, dst)
    print(f"{dest.name}: {dest.stat().st_size/1e6:.0f} MB")
    return dest

def layout(path):
    """Chunk shape, chunk size, and codec for every variable in the file."""
    tree = xr.open_datatree(path, engine="h5netcdf", phony_dims="access", decode_times=False)
    rows = []
    for node in tree.subtree:
        for name, v in node.ds.data_vars.items():
            ch = v.encoding.get("chunksizes")
            rows.append({"variable": f"{node.path}/{name}", "shape": tuple(v.shape),
                         "MB": round(v.nbytes / 1e6, 2),
                         "chunks": tuple(ch) if ch else None,
                         "chunk_MB": round(v.dtype.itemsize * int(np.prod(ch)) / 1e6, 3) if ch else None,
                         "codec": v.encoding.get("compression"),
                         "shuffle": bool(v.encoding.get("shuffle"))})
    return pd.DataFrame(rows).sort_values("MB", ascending=False)

## METimage (VII) level 1B radiances

The full walkthrough. Each later instrument repeats these same five steps.

First, the newest granule and its package contents. Expect one `.nc` and two
XML sidecars: the zip exists to carry those sidecars.

In [16]:
spec = PRODUCTS["METimage (VII)"]
prod, entries = latest(spec["collection"])
nc = next(e for e in entries if e.endswith(".nc"))
url = entry_url(prod, nc)

W_XX-EUMETSAT-Darmstadt,SAT,SGA1-VII-1B-RAD_C_EUMT_20260916205124_G_O_20260916200259_20260916200400_C_N_T__ 
  files: ['W_XX-EUMETSAT-Darmstadt,SAT,SGA1-VII-1B-RAD_C_EUMT_20260916205124_G_O_20260916200259_20260916200400_C_N_T__.nc', 'EOPMetadata.xml', 'manifest.xml']


Check range support on the per-file URL. Without 206 responses here, nothing
else in this notebook is possible and the product is download-only.

In [17]:
ranges_ok(url)

first bytes: 894844460d0a1a0a  (894844... means HDF5/netCDF-4)
range status: {'head': 206, 'tail': 206}


True

Open the granule over HTTP. This reads metadata by range request and defers
everything else, so it should finish in seconds even though the file is over
100 MB. Then pull a 64x64 slice of one radiance channel to confirm that
subsetting works and to feel the per-read latency.

In [18]:
%time tree = open_remote(url)
tree

CPU times: user 1.21 s, sys: 152 ms, total: 1.36 s
Wall time: 53.7 s


<xarray.DataTree>
Group: /
│   Attributes: (12/21)
│       title:                   VII L1B Radiances
│       Conventions:             CF-1.6
│       metadata_conventions:    Unidata Dataset Discovery v1.0
│       product_name:            W_XX-EUMETSAT-Darmstadt,SAT,SGA1-VII-1B-RAD_C_EU...
│       summary:                 VII/METimage L1B top of the atmosphere radiances
│       doi:                     
│       ...                      ...
│       sensing_start_time_utc:  2026-09-16 20:02:59.643
│       sensing_end_time_utc:    2026-09-16 20:04:00.134
│       environment:             Operational
│       references:              www.eumetsat.int
│       orbit_start:             5681
│       orbit_end:               5681
├── Group: /status
│   ├── Group: /status/satellite
│   │       Dimensions:                   ()
│   │       Data variables: (12/24)
│   │           epoch_time_utc            datetime64[ns] 8B ...
│   │           semi_major_axis           float64 8B ...
│   │           eccentricity              float64 8B ...
│   │           inclination               float64 8B ...
│   │           perigee_argument          float64 8B ...
│   │           right_ascension           float64 8B ...
│   │           ...                        ...
│   │           z_velocity                float64 8B ...
│   │           yaw_error                 float64 8B ...
│   │           roll_error                float64 8B ...
│   │           pitch_error               float64 8B ...
│   │           leap_second_time_utc      datetime64[ns] 8B ...
│   │           leap_second_value         float32 4B ...
│   ├── Group: /status/instrument
│   │       Dimensions:              (mode_items: 1)
│   │       Dimensions without coordinates: mode_items
│   │       Data variables:
│   │           mode_start_time_utc  (mode_items) datetime64[ns] 8B ...
│   │           mode_end_time_utc    (mode_items) datetime64[ns] 8B ...
│   │           instrument_mode      (mode_items) <U4 16B ...
│   └── Group: /status/processing
│           Dimensions:            ()
│           Data variables:
│               creation_time_utc  datetime64[ns] 8B ...
│           Attributes:
│               processor_name:              VII_L1B
│               processor_version:           1.0
│               processing_mode:             NRT
│               format_version:              6.0
│               auxiliary_data_version:      EUM/LEO-EPSSG/SPE/14/777147 v4A
│               pgs_reference_and_version:   EUM/LEO-EPSSG/DOC/14/746628 v5
│               pfs_reference_and_version:   EUM/LEO-EPSSG/SPE/14/777138 v5
│               atbd_reference_and_version:  EUM/LEO-EPSSG/DOC/13/702485 v4
│               source:                      ['SGA1_VII_1B_AUX_LMDB___S20250813000000Z_Ex...
├── Group: /data
│   ├── Group: /data/measurement_data
│   │       Dimensions:              (num_tie_points_alt: 140, num_tie_points_act: 394,
│   │                                 num_lines: 840, num_pixels: 3144, num_scans: 35)
│   │       Dimensions without coordinates: num_tie_points_alt, num_tie_points_act,
│   │                                       num_lines, num_pixels, num_scans
│   │       Data variables: (12/32)
│   │           latitude             (num_tie_points_alt, num_tie_points_act) float64 441kB ...
│   │           longitude            (num_tie_points_alt, num_tie_points_act) float64 441kB ...
│   │           delta_lat_N_dem      (num_lines, num_pixels) float32 11MB ...
│   │           delta_lon_E_dem      (num_lines, num_pixels) float32 11MB ...
│   │           solar_zenith         (num_tie_points_alt, num_tie_points_act) float64 441kB ...
│   │           solar_azimuth        (num_tie_points_alt, num_tie_points_act) float64 441kB ...
│   │           ...                   ...
│   │           vii_6725             (num_lines, num_pixels) float32 11MB ...
│   │           vii_7325             (num_lines, num_pixels) float32 11MB ...
│   │           vii_8540             (num_lines, num_pixels) float32 11MB

In [19]:
rad = tree["data/measurement_data"].ds
channel = [v for v in rad.data_vars if "radiance" in v.lower() or "vii_" in v.lower()][0]
%time sample = rad[channel][:64, :64].values
print(channel, sample.shape, sample.dtype)

CPU times: user 66.8 ms, sys: 18.2 ms, total: 85 ms
Wall time: 3.44 s
vii_443 (64, 64) float32


Download the file once and read its storage layout. Three things to look for
in the table: whether large variables are chunked at all, whether chunk sizes
land anywhere near useful targets (1 to 4 MB for interactive reads, 1 to
16 MB for agentic subsetting, 32 to 64 MB for training), and whether any
codec is set. A `codec` column full of `None` is the quantitative version of
the feedback that compression belongs inside the netCDF.

In [20]:
local = download(prod, nc)
L = layout(local)
L.head(15)

W_XX-EUMETSAT-Darmstadt,SAT,SGA1-VII-1B-RAD_C_EUMT_20260916205124_G_O_20260916200259_20260916200400_C_N_T__.nc: 119 MB


,variable,shape,MB,chunks,chunk_MB,codec,shuffle
46,/data/measurement_data/vii_668,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
45,/data/measurement_data/vii_555,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
34,/data/measurement_data/delta_lat_N_dem,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
35,/data/measurement_data/delta_lon_E_dem,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
50,/data/measurement_data/vii_914,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
48,/data/measurement_data/vii_763,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
49,/data/measurement_data/vii_865,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
47,/data/measurement_data/vii_752,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
44,/data/measurement_data/vii_443,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
60,/data/measurement_data/vii_8540,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False


In [21]:
print("compressed variables:", int(L.codec.notna().sum()), "of", len(L))
print("logical size:", round(L.MB.sum()), "MB   on disk:", round(local.stat().st_size/1e6), "MB")
print("chunk sizes (MB):", L.chunk_MB.describe()[["min", "50%", "max"]].round(3).to_dict())

compressed variables: 0 of 115
logical size: 235 MB   on disk: 119 MB
chunk sizes (MB): {'min': 0.0, '50%': 0.013, 'max': 0.013}


Measure what internal compression would buy. This compresses one real chunk
on local CPU. On float radiances, deflate level 1 with byte shuffle is the
cheap, universally readable option; zstd does the same job faster. Ratios
above about 1.5x make a solid case.

In [22]:
import zstandard as zstd
row = L.dropna(subset=["chunks"]).iloc[0]
arr = np.ascontiguousarray(
    tree[str(Path(row.variable).parent)].ds[Path(row.variable).name][:512, :512].values)
raw, shuf = arr.tobytes(), arr.view(np.uint8).reshape(-1, arr.dtype.itemsize).T.tobytes()

print(row.variable, arr.shape, arr.dtype)
print(f"deflate-1: {len(raw)/len(zlib.compress(raw,1)):.2f}x, "
      f"with shuffle {len(raw)/len(zlib.compress(shuf,1)):.2f}x")
c = zstd.ZstdCompressor(level=3)
print(f"zstd-3:    {len(raw)/len(c.compress(raw)):.2f}x, "
      f"with shuffle {len(raw)/len(c.compress(shuf)):.2f}x")

/data/measurement_data/vii_668 (512, 512) float32
deflate-1: 1.93x, with shuffle 1.35x
zstd-3:    1.59x, with shuffle 1.35x


## MWS

Microwave sounder, 24 channels, small granules. The same steps usually run in
under a minute end to end.

In [29]:
spec = PRODUCTS["MWS"]
if spec.get("collection"):
    prod, entries = latest(spec["collection"])
    datafile = next(e for e in entries if not e.lower().endswith((".xml", ".txt")))
    url = entry_url(prod, datafile)
else:
    url = spec["url"]                     # direct test-data link
    assert url, "paste this product's link from the test-data page into PRODUCTS"

ranges_ok(url)

W_XX-EUMETSAT-Darmstadt,SAT,SGA1-MWS-1B-RAD_C_EUMT_20260916212055_G_O_20260916203857_20260916204158_O_N____ 
  files: ['W_XX-EUMETSAT-Darmstadt,SAT,SGA1-MWS-1B-RAD_C_EUMT_20260916212055_G_O_20260916203857_20260916204158_O_N____.nc', 'EOPMetadata.xml', 'manifest.xml']
first bytes: 894844460d0a1a0a  (894844... means HDF5/netCDF-4)
range status: {'head': 206, 'tail': 206}


True

In [30]:
tree = open_remote(url)
local = download(prod, datafile) if spec.get("collection") else fetch(url)
L = layout(local)
print("compressed:", int(L.codec.notna().sum()), "of", len(L),
      "  median chunk MB:", L.chunk_MB.median())
L.head(10)

W_XX-EUMETSAT-Darmstadt,SAT,SGA1-MWS-1B-RAD_C_EUMT_20260916212055_G_O_20260916203857_20260916204158_O_N____.nc: 3 MB
compressed: 0 of 167   median chunk MB: 0.1075


,variable,shape,MB,chunks,chunk_MB,codec,shuffle
130,/data/calibration/mws_toa_radiance,"(80, 95, 24)",1.46,"(28, 95, 24)",0.511,None,False
108,/data/calibration/mws_toa_brightness_temperature,"(80, 95, 24)",1.46,"(28, 95, 24)",0.511,None,False
136,/data/measurement/mws_earth_view_counts,"(80, 95, 24)",0.73,"(57, 95, 24)",0.520,None,False
140,/data/processing_information/mws_radiance_flag,"(80, 95, 24)",0.18,"(80, 95, 24)",0.182,None,False
162,/data/processing_information/mws_brightnesstem...,"(80, 95, 24)",0.18,"(80, 95, 24)",0.182,None,False
102,/data/navigation/mws_surface_type,"(80, 95, 2)",0.12,None,NaN,None,False
99,/data/navigation/mws_solar_azimuth_angle,"(80, 95)",0.06,None,NaN,None,False
98,/data/navigation/mws_satellite_zenith_angle,"(80, 95)",0.06,None,NaN,None,False
104,/data/navigation/mws_lat,"(80, 95)",0.06,None,NaN,None,False
135,/data/measurement/mws_earth_view_counts_os_stdev,"(80, 95, 2)",0.06,None,NaN,None,False


## Radio Occultation

GRAS-2 has a Data Store collection, and its title says netCDF. The
`ranges_ok` call prints the first bytes as a check anyway: `894844` opens
like the others, `425546` spells BUFR, which has no internal chunking to
assess and would end this section early as its own finding.


In [32]:
spec = PRODUCTS["Radio Occultation"]
if spec.get("collection"):
    prod, entries = latest(spec["collection"])
    datafile = next(e for e in entries if not e.lower().endswith((".xml", ".txt")))
    url = entry_url(prod, datafile)
else:
    url = spec["url"]                     # direct test-data link
    assert url, "paste this product's link from the test-data page into PRODUCTS"

ranges_ok(url)

W_XX-EUMETSAT-Darmstadt,SAT,SGA1-RO_-1B-BND_C_EUMT_20260916211844_G_O_20260916202243_20260916202900_O_N_G20 
  files: ['W_XX-EUMETSAT-Darmstadt,SAT,SGA1-RO_-1B-BND_C_EUMT_20260916211844_G_O_20260916202243_20260916202900_O_N_G20.nc', 'EOPMetadata.xml', 'manifest.xml']
first bytes: 894844460d0a1a0a  (894844... means HDF5/netCDF-4)
range status: {'head': 206, 'tail': 206}


True

### Read Efficiency
This section tests the read efficiency for hosted vs local read for the "EO:EUM:DAT:0464" product.

In [56]:
import time, urllib.parse
from pathlib import Path
import xarray as xr, fsspec, requests

LOCAL = Path("data/W_XX-EUMETSAT-Darmstadt,SAT,SGA1-VII-1B-RAD_C_EUMT_20260916205124_G_O_20260916200259_20260916200400_C_N_T__.nc")
enc = lambda s: urllib.parse.quote(s, safe="")

pid  = LOCAL.name[:-3]                       # product id = filename minus .nc
base = ("https://api.eumetsat.int/data/download/1.0.0/collections/"
        f"{enc('EO:EUM:DAT:0464')}/products/{enc(pid)}")
URL  = f"{base}/entry?name={enc(LOCAL.name)}"

def remote_size(u):
    r = requests.get(u, headers={**auth(), "Range": "bytes=0-0"}, stream=True, timeout=60); r.close()
    return int(r.headers["Content-Range"].split("/")[-1]) if r.status_code == 206 \
           else int(r.headers["Content-Length"])

# ---- pick the target variable from the local copy -------------------------
lt  = xr.open_datatree(LOCAL, engine="h5netcdf", phony_dims="access", decode_times=False)
rad = lt["data/measurement_data"].ds
var = "vii_443"
ny, nx = rad[var].shape[:2]
sl = (slice(ny//2, ny//2+64), slice(nx//2, nx//2+64))      # mid-swath, not a corner
chunks = rad[var].encoding.get("chunksizes")
print(f"variable : {var} {rad[var].shape} {rad[var].dtype}")
print(f"chunks   : {chunks}  codec={rad[var].encoding.get('compression')}")
print(f"file     : {LOCAL.stat().st_size/1e6:.0f} MB\n")

# ---- local baseline -------------------------------------------------------
t0 = time.perf_counter(); a = rad[var][sl].values; t_local = time.perf_counter()-t0
wanted = a.nbytes
print(f"local slice : {wanted/1e3:.0f} KB in {t_local*1e3:.0f} ms\n")

bs = 8*1024*1024
fs = fsspec.filesystem("https", client_kwargs={"headers": auth()},
                       encoded=True, skip_instance_cache=True)
f  = fs.open(URL, block_size=bs, size=remote_size(URL))

t0 = time.perf_counter()
rt = xr.open_datatree(f, engine="h5netcdf", phony_dims="access", decode_times=False)
t_open = time.perf_counter() - t0
open_bytes, open_reqs = f.cache.total_requested_bytes, f.cache.miss_count

t0 = time.perf_counter()
b = rt["data/measurement_data"].ds[var][sl].values
t_read = time.perf_counter() - t0
read_bytes = f.cache.total_requested_bytes - open_bytes
read_reqs  = f.cache.miss_count - open_reqs

print(f"open  : {open_bytes/1e6:6.1f} MB in {open_reqs:3d} reads, {t_open:5.1f}s")
print(f"slice : {read_bytes/1e6:6.1f} MB in {read_reqs:3d} reads, {t_read:5.1f}s"
      f"   -> {read_bytes/wanted:.0f}x amplification for {wanted/1e3:.0f} KB wanted")
print(f"total : {(open_bytes+read_bytes)/1e6:6.1f} MB of a "
      f"{LOCAL.stat().st_size/1e6:.0f} MB file")

variable : vii_443 (840, 3144) float32
chunks   : (1, 3144)  codec=None
file     : 119 MB

local slice : 16 KB in 8 ms

open  :   75.5 MB in  12 reads,  21.7s
slice :   16.7 MB in   2 reads,   6.5s   -> 1019x amplification for 16 KB wanted
total :   92.2 MB of a 119 MB file
